<!--
---
PURPOSE: Convert pose predictions to features and motifs, with automated SLEAP inference.
REQUIRES:
  - outputs/video/video_assets.parquet (or SLEAP CSV exports)
PRODUCES:
  - outputs/pose/session_{id}_pose_predictions.parquet
  - outputs/pose/session_{id}_pose_features.parquet
  - outputs/pose/session_{id}_motifs.parquet
WHAT_NEXT: notebooks/08_Neural_Behavior_Fusion_and_Modeling.ipynb
---
-->

# 07 Pose to Motifs Feature Engineering

**Purpose**
Convert pose predictions to rich behavioral features and motifs.

**New in this version:**
- **Auto-discover** SLEAP CSV exports and .slp prediction files (no manual path pasting)
- **Automated SLEAP inference** on locally-cached videos (if a trained model exists)
- **Rich feature extraction**: per-keypoint velocity, acceleration, body length, head angle, stillness
- **Confidence filtering**: drop low-confidence detections before feature extraction
- **Active learning**: suggest which frames to label next for maximum model improvement

**Produces**
- `outputs/pose/session_{id}_pose_predictions.parquet`
- `outputs/pose/session_{id}_pose_features.parquet`
- `outputs/pose/session_{id}_motifs.parquet`

**What to run next**
- `notebooks/08_Neural_Behavior_Fusion_and_Modeling.ipynb`

## Environment Setup

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()      
ROOT = ROOT.parent              
SRC  = ROOT / "src"           

# put repo + src on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


#print("ROOT:", ROOT)
#print("SRC :", SRC, "| exists:", SRC.exists())
#print("sys.path[0:3]:", sys.path[:3])

## Step 1: Auto-discover pose data

We look for pose predictions in three places (in priority order):
1. **Existing `.parquet` predictions** from prior runs
2. **SLEAP CSV exports** (from manual labeling + inference)
3. **SLEAP `.slp` prediction files** (from automated inference)

If nothing is found and a trained SLEAP model exists, we can run inference automatically.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from config import get_config, make_provenance
from io_sessions import load_sessions_csv
from io_video import load_frame_times
from features_pose import (
    export_pose_predictions_from_sleap_csv,
    derive_pose_features,
    filter_by_confidence,
)
from pose_inference import (
    auto_discover_sleap_csvs,
    discover_sleap_models,
    run_batch_inference,
    slp_to_parquet,
    suggest_frames_to_label,
)
from timebase import write_parquet_with_timebase

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()

# --- Configuration ---
CONFIDENCE_THRESHOLD = 0.3  # drop keypoints below this confidence
CAMERA = "side"             # primary camera for pose
RUN_INFERENCE = True        # set False to skip automated inference

print(f"Sessions to process: {SESSION_IDS}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Camera: {CAMERA}")

# --- Auto-discover existing predictions ---
for sid in SESSION_IDS:
    pose_path = cfg.outputs_dir / "pose" / f"session_{sid}_pose_predictions.parquet"

    if pose_path.exists():
        print(f"\n[session {sid}] Found existing predictions: {pose_path}")
        continue

    # Try auto-discovering SLEAP CSV exports
    csvs = auto_discover_sleap_csvs(session_id=sid)
    if csvs:
        csv_info = csvs[0]  # use first match
        print(f"\n[session {sid}] Auto-discovered SLEAP CSV: {csv_info['path']}")
        cam = csv_info.get("camera") or CAMERA
        out = export_pose_predictions_from_sleap_csv(
            Path(csv_info["path"]),
            session_id=sid,
            camera=cam,
        )
        print(f"  -> Wrote predictions: {out}")
        continue

    # Try finding .slp prediction files
    slp_candidates = list((cfg.outputs_dir / "pose" / "predictions").glob(f"*{sid}*.slp"))
    if slp_candidates:
        print(f"\n[session {sid}] Found .slp predictions: {slp_candidates[0]}")
        n = slp_to_parquet(slp_candidates[0], sid, CAMERA, output_path=pose_path)
        print(f"  -> Converted {n} frames to parquet")
        continue

    print(f"\n[session {sid}] No predictions found yet.")

print("\nDiscovery complete.")

In [ ]:
# --- Optional: Run automated SLEAP inference on cached videos ---
# This requires a trained SLEAP model. With your ~150 labeled frames,
# you can train one via: sleap-train <config.json> <labels.slp>
# Then place the model in pose_projects/ or data/sleap_models/

if RUN_INFERENCE:
    models = discover_sleap_models()
    if models:
        print(f"Found {len(models)} SLEAP model(s):")
        for m in models:
            print(f"  - {m['path']} ({m.get('type', 'unknown')})")

        # Only run on sessions that don't already have predictions
        missing_sids = [
            sid for sid in SESSION_IDS
            if not (cfg.outputs_dir / "pose" / f"session_{sid}_pose_predictions.parquet").exists()
        ]
        if missing_sids:
            print(f"\nRunning inference on {len(missing_sids)} sessions without predictions...")
            inference_results = run_batch_inference(
                session_ids=missing_sids,
                cameras=[CAMERA],
                model_paths=[models[0]["path"]],
                skip_existing=True,
            )
            print(inference_results[["session_id", "camera", "n_frames", "status"]])
        else:
            print("\nAll sessions already have predictions.")
    else:
        print("No SLEAP models found. To enable automated inference:")
        print("  1. Train a model: sleap-train <config.json> <your_labels.slp>")
        print("  2. Place the model directory in pose_projects/ or data/sleap_models/")
        print("  3. Re-run this cell")

## Step 2: Feature extraction and motif discovery

For each session with predictions, we:
1. **Filter by confidence** — drop low-quality keypoint detections
2. **Extract rich features** — velocity, acceleration, body length, head angle, stillness
3. **Discover motifs** — cluster behavioral features into discrete states (k-means or HMM)

In [ ]:
from motifs import motifs_kmeans, motifs_hmm
from viz import plot_motif_transition

# --- Configuration ---
N_MOTIF_CLUSTERS = 8         # number of behavioral motifs
MOTIF_METHOD = "kmeans"      # "kmeans" or "hmm"

for sid in SESSION_IDS:
    pose_path = cfg.outputs_dir / "pose" / f"session_{sid}_pose_predictions.parquet"

    if not pose_path.exists():
        print(f"\n[session {sid}] SKIP - no predictions found")
        continue

    print(f"\n{'='*60}")
    print(f"[session {sid}] Processing pose predictions")
    print(f"{'='*60}")

    # Load predictions
    pose_df = pd.read_parquet(pose_path)
    if "camera" in pose_df.columns:
        pose_df = pose_df[pose_df["camera"] == CAMERA].reset_index(drop=True)
    print(f"  Raw predictions: {len(pose_df)} rows, {len(pose_df.columns)} columns")

    # Show available keypoints
    kp_cols = [c for c in pose_df.columns if c.endswith("_x")]
    keypoint_names = [c[:-2] for c in kp_cols]
    print(f"  Keypoints found: {keypoint_names}")

    # Show confidence stats (if available)
    score_cols = [c for c in pose_df.columns if c.endswith("_score")]
    if score_cols:
        mean_scores = pose_df[score_cols].mean()
        print(f"  Mean confidence per keypoint:")
        for col, score in mean_scores.items():
            print(f"    {col}: {score:.3f}")

    # Filter by confidence
    if CONFIDENCE_THRESHOLD > 0:
        before = pose_df[kp_cols].notna().sum().sum()
        pose_df = filter_by_confidence(pose_df, CONFIDENCE_THRESHOLD, method="nan")
        after = pose_df[kp_cols].notna().sum().sum()
        pct_kept = 100 * after / before if before > 0 else 0
        print(f"  Confidence filter: kept {pct_kept:.1f}% of keypoint detections")

    # Extract rich features
    features = derive_pose_features(pose_df, confidence_threshold=0)  # already filtered above
    if features is None or features.empty:
        print(f"  WARNING: No features extracted for session {sid}")
        continue

    print(f"  Features extracted: {list(features.columns)}")
    print(f"  Feature shape: {features.shape}")

    # Save features
    features_path = cfg.outputs_dir / "pose" / f"session_{sid}_pose_features.parquet"
    write_parquet_with_timebase(
        features, features_path,
        provenance=make_provenance(sid, "nwb"),
        required_columns=["t"],
    )

    # Discover motifs
    if MOTIF_METHOD == "hmm":
        motifs_df = motifs_hmm(features, n_states=N_MOTIF_CLUSTERS)
    else:
        motifs_df = motifs_kmeans(features, n_clusters=N_MOTIF_CLUSTERS)

    motifs_path = cfg.outputs_dir / "pose" / f"session_{sid}_motifs.parquet"
    write_parquet_with_timebase(
        motifs_df, motifs_path,
        provenance=make_provenance(sid, "nwb"),
        required_columns=["t", "motif_id"],
    )
    print(f"  Motifs: {motifs_df['motif_id'].nunique()} clusters, {len(motifs_df)} time bins")

    # Visualize motif transitions
    plot_motif_transition(motifs_df)

    # Feature summary statistics
    print(f"\n  Feature summary:")
    for col in ["pose_speed", "body_length", "head_angle", "is_still"]:
        if col in features.columns:
            vals = features[col].dropna()
            print(f"    {col}: mean={vals.mean():.3f}, std={vals.std():.3f}, range=[{vals.min():.3f}, {vals.max():.3f}]")

    print(f"  Saved: {features_path}")
    print(f"  Saved: {motifs_path}")

In [ ]:
# --- Active learning: suggest frames to label next ---
# If you have .slp prediction files, this tells you which frames
# the model is LEAST confident about -- labeling those gives you
# the biggest improvement per frame labeled.

print("Active learning suggestions:")
print("=" * 40)

for sid in SESSION_IDS:
    slp_candidates = list((cfg.outputs_dir / "pose" / "predictions").glob(f"*{sid}*.slp"))
    if not slp_candidates:
        print(f"[session {sid}] No .slp predictions found (skip active learning)")
        continue

    try:
        suggestions = suggest_frames_to_label(
            slp_candidates[0],
            n_suggestions=20,
            strategy="spread",
        )
        print(f"[session {sid}] Suggested frames to label next (lowest confidence, spread across session):")
        print(f"  Frame indices: {suggestions.tolist()}")
        print(f"  To label these, open SLEAP GUI with:")
        print(f"    sleap-label {slp_candidates[0]}")
    except Exception as e:
        print(f"[session {sid}] Active learning error: {e}")

print("\nNotebook 07 complete. Run Notebook 08 for neural-behavior fusion.")